**© Copyright AIDENTIFY. All rights reserved.**

본 자료는 **멀티캠퍼스 LLM 파인튜닝 과정** 수강생을 위해 제작되었으며, 강의 목적으로만 사용 가능합니다.  
무단 복제, 배포, 상업적 이용을 금지합니다.

---

# Session 33: 프로젝트 기획 - 도메인 특화 sLLM 파인튜닝

## 🎯 프로젝트 개요 및 목표

이 노트북에서는 **도메인 특화 sLLM 파인튜닝 프로젝트**를 기획합니다.

### 프로젝트 목표
- 특정 도메인에 최적화된 소형 언어모델(sLLM)을 파인튜닝합니다
- 데이터 수집 → 정제 → 학습 → 평가 → 배포의 **전체 파이프라인**을 경험합니다
- 실제 비즈니스 문제를 해결할 수 있는 **실용적인 AI 시스템**을 구축합니다

### 프로젝트 진행 순서

| 단계 | 노트북 | 내용 |
|------|--------|------|
| 1단계 | **32_project_planning** (현재) | 기획 및 계획 수립 |
| 2단계 | 33_project_data_pipeline | 데이터 파이프라인 구축 |
| 3단계 | 34_project_training | 모델 학습 수행 |
| 4단계 | 35_project_evaluation | 성능 평가 및 개선 |
| 5단계 | 36_project_deployment | 배포 및 어플리케이션 |

### 실습 환경
- **GPU**: NVIDIA RTX 4060 (8GB VRAM)
- **권장 모델**: Qwen2.5-1.5B-Instruct / Qwen2.5-3B-Instruct
- **방법론**: LoRA (1.5B~3B) / QLoRA (7B)

---
## 1️⃣ 도메인 선택 가이드

아래 표를 참고하여 프로젝트 도메인을 선택하세요.

| 도메인 | 예시 태스크 | 데이터 소스 | 난이도 | RTX 4060 적합도 |
|--------|------------|------------|--------|----------------|
| 🏥 **의료/건강** | 증상→질환 매핑, 건강 상담 | 의학 Q&A, 논문 요약 | ⭐⭐⭐ | ✅ (1.5B~3B) |
| ⚖️ **법률** | 법률 용어 해석, 판례 요약 | 법률 상담 데이터, 판례 | ⭐⭐⭐ | ✅ (1.5B~3B) |
| 💰 **금융** | 금융 상품 설명, 투자 조언 | 금융 FAQ, 뉴스 요약 | ⭐⭐ | ✅ (1.5B~3B) |
| 📚 **교육** | 학습 도우미, 문제 풀이 | 교과서, Q&A 데이터 | ⭐⭐ | ✅ (1.5B~3B) |
| 🎧 **고객서비스** | FAQ 응답, 불만 처리 | 고객 상담 로그 | ⭐ | ✅ (1.5B~3B) |
| 💻 **코딩** | 코드 생성, 디버깅 | 코드 데이터셋 | ⭐⭐ | ✅ (1.5B~3B) |
| 🍳 **요리/레시피** | 레시피 추천, 재료 대체 | 요리 데이터 | ⭐ | ✅ (1.5B~3B) |
| 🎮 **게임** | NPC 대화, 게임 가이드 | 게임 위키, 대화 로그 | ⭐⭐ | ✅ (1.5B~3B) |

### 도메인 선택 기준
- **데이터 접근성**: 해당 도메인의 데이터를 충분히 확보할 수 있는가?
- **평가 용이성**: 모델 출력의 품질을 객관적으로 평가할 수 있는가?
- **실용적 가치**: 실제로 사용될 수 있는 서비스인가?
- **개인 관심도**: 꾸준히 작업할 동기가 있는가?

In [ ]:
# 도메인 선택 도우미
# 각 도메인에 대한 정보를 확인하고, 적합한 도메인을 선택하세요

domain_info = {
    "의료/건강": {
        "tasks": ["증상 분석", "건강 상담", "의학 용어 설명", "약물 상호작용 안내"],
        "data_sources": ["의학 Q&A 사이트", "건강 상담 데이터", "의학 교과서 요약"],
        "eval_metrics": ["의학적 정확성", "안전성 (유해 조언 방지)", "이해하기 쉬운 설명"],
        "sample_size": "500~2000개 권장"
    },
    "법률": {
        "tasks": ["법률 용어 해석", "판례 요약", "법률 상담 초안", "계약서 검토"],
        "data_sources": ["법률 상담 데이터", "판례 데이터베이스", "법률 FAQ"],
        "eval_metrics": ["법률적 정확성", "조항 인용 정확도", "명확성"],
        "sample_size": "500~2000개 권장"
    },
    "금융": {
        "tasks": ["금융 상품 설명", "투자 정보 요약", "재무제표 분석", "리스크 안내"],
        "data_sources": ["금융 FAQ", "투자 교육 자료", "금융 뉴스"],
        "eval_metrics": ["정보 정확성", "리스크 고지 포함 여부", "명확성"],
        "sample_size": "500~1500개 권장"
    },
    "교육": {
        "tasks": ["개념 설명", "문제 풀이 도우미", "학습 계획 수립", "퀴즈 생성"],
        "data_sources": ["교과서 내용", "교육 Q&A", "강의 자료"],
        "eval_metrics": ["설명의 정확성", "교육적 적절성", "학습자 수준 맞춤"],
        "sample_size": "500~1500개 권장"
    },
    "고객서비스": {
        "tasks": ["FAQ 자동 응답", "불만 처리", "주문 안내", "기술 지원"],
        "data_sources": ["고객 상담 로그", "FAQ 데이터", "제품 매뉴얼"],
        "eval_metrics": ["응답 적절성", "고객 만족도", "문제 해결률"],
        "sample_size": "300~1000개 권장"
    },
    "코딩": {
        "tasks": ["코드 생성", "버그 수정", "코드 설명", "리팩토링 제안"],
        "data_sources": ["코드 Q&A", "GitHub 이슈", "코딩 튜토리얼"],
        "eval_metrics": ["코드 실행 가능성", "정확성", "코드 품질"],
        "sample_size": "500~2000개 권장"
    }
}

# 도메인 정보 출력
for domain, info in domain_info.items():
    print(f"\n{'='*50}")
    print(f"📌 {domain}")
    print(f"{'='*50}")
    print(f"  태스크: {', '.join(info['tasks'])}")
    print(f"  데이터 소스: {', '.join(info['data_sources'])}")
    print(f"  평가 기준: {', '.join(info['eval_metrics'])}")
    print(f"  권장 데이터 수: {info['sample_size']}")

---
## 2️⃣ 문제 정의 템플릿

프로젝트의 핵심은 **명확한 문제 정의**입니다. 아래 템플릿을 채워보세요.

### 문제 정의의 3요소
1. **문제(Problem)**: 무엇을 해결하려고 하는가?
2. **목표(Goal)**: 어떤 결과를 달성하려고 하는가?
3. **평가 기준(Criteria)**: 어떻게 성공을 측정할 것인가?

In [ ]:
# =====================================================
# 📝 문제 정의 템플릿
# TODO: 아래 항목을 모두 채워주세요
# =====================================================

project_definition = {
    # -------------------------------------------------
    # 1. 문제 정의
    # -------------------------------------------------
    "problem": {
        "domain": "",           # TODO: 도메인 입력 (예: "의료/건강")
        "target_task": "",      # TODO: 해결할 구체적 태스크 (예: "증상 기반 질환 안내")
        "pain_point": "",       # TODO: 현재 문제점 (예: "일반 LLM은 의학 용어를 부정확하게 사용")
        "target_user": "",      # TODO: 대상 사용자 (예: "건강 정보를 찾는 일반인")
    },
    
    # -------------------------------------------------
    # 2. 목표
    # -------------------------------------------------
    "goal": {
        "primary": "",          # TODO: 주요 목표 (예: "의료 상담 품질 향상")
        "secondary": "",        # TODO: 부가 목표 (예: "응답 시간 단축")
        "success_criteria": "", # TODO: 성공 기준 (예: "전문가 평가 80% 이상 적절")
    },
    
    # -------------------------------------------------
    # 3. 평가 기준
    # -------------------------------------------------
    "evaluation": {
        "auto_metrics": [],     # TODO: 자동 평가 지표 (예: ["ROUGE-L", "BLEU"])
        "llm_judge": True,      # LLM-as-a-Judge 사용 여부
        "human_eval": True,     # 사람 평가 포함 여부
        "eval_dimensions": [],  # TODO: 평가 차원 (예: ["정확성", "유용성", "안전성"])
    }
}

# 입력 확인
def validate_definition(definition):
    """문제 정의 완성도를 확인합니다."""
    missing = []
    for section, items in definition.items():
        for key, value in items.items():
            if isinstance(value, str) and value == "":
                missing.append(f"{section}.{key}")
            elif isinstance(value, list) and len(value) == 0:
                missing.append(f"{section}.{key}")
    
    if missing:
        print("⚠️ 아직 채우지 않은 항목이 있습니다:")
        for item in missing:
            print(f"   - {item}")
    else:
        print("✅ 모든 항목이 작성되었습니다!")
    return len(missing) == 0

validate_definition(project_definition)

### 💡 문제 정의 예시

```python
# 예시: 고객서비스 챗봇
project_definition = {
    "problem": {
        "domain": "고객서비스",
        "target_task": "온라인 쇼핑몰 고객 문의 자동 응답",
        "pain_point": "일반 LLM은 회사 정책/상품 정보를 모르고, 환각 답변이 많음",
        "target_user": "쇼핑몰 고객 및 CS 담당자",
    },
    "goal": {
        "primary": "회사 정책 기반 정확한 고객 응답 생성",
        "secondary": "CS 담당자 업무 부하 30% 감소",
        "success_criteria": "LLM-as-a-Judge 정확성 점수 4.0/5.0 이상",
    },
    "evaluation": {
        "auto_metrics": ["ROUGE-L", "BERTScore"],
        "llm_judge": True,
        "human_eval": True,
        "eval_dimensions": ["정확성", "친절도", "정책 준수", "완결성"],
    }
}
```

---
## 3️⃣ 데이터 전략 수립

파인튜닝 프로젝트의 성패는 **데이터 품질**에 달려있습니다.

### 데이터 전략 고려사항

| 항목 | 선택지 | 권장 |
|------|--------|------|
| 수집 방법 | 크롤링 / 공개 데이터셋 / 합성 생성 | 혼합 사용 |
| 데이터 양 | 최소 300 ~ 최대 5000+ | 500~2000 |
| 데이터 형식 | Alpaca / ShareGPT / Custom | Alpaca 권장 |
| 품질 관리 | 수동 검수 / LLM 필터링 / 규칙 기반 | LLM + 수동 |
| 증강 방법 | 패러프레이징 / 역번역 / Distillation | Distillation |

In [ ]:
# =====================================================
# 📝 데이터 전략 템플릿
# TODO: 아래 항목을 모두 채워주세요
# =====================================================

data_strategy = {
    # -------------------------------------------------
    # 1. 데이터 수집
    # -------------------------------------------------
    "collection": {
        "sources": [],          # TODO: 데이터 소스 (예: ["HuggingFace 데이터셋", "웹 크롤링", "합성 생성"])
        "hf_datasets": [],      # TODO: HuggingFace 데이터셋 이름 (예: ["kor_medical_qa"])
        "target_count": 0,      # TODO: 목표 데이터 수 (예: 1000)
        "language": "ko",       # 데이터 언어
    },
    
    # -------------------------------------------------
    # 2. 데이터 형식
    # -------------------------------------------------
    "format": {
        # Alpaca 형식: {"instruction": ..., "input": ..., "output": ...}
        # ShareGPT 형식: {"conversations": [{"from": "human", "value": ...}, ...]}
        "type": "",             # TODO: "alpaca" 또는 "sharegpt"
        "max_length": 1024,     # 최대 토큰 길이 (RTX 4060 기준)
    },
    
    # -------------------------------------------------
    # 3. 품질 관리
    # -------------------------------------------------
    "quality": {
        "dedup_method": "",     # TODO: 중복 제거 방법 (예: "exact_match", "similarity")
        "filter_rules": [],     # TODO: 필터링 규칙 (예: ["최소 20자", "한국어 비율 80% 이상"])
        "llm_review": True,     # LLM으로 품질 검수
    },
    
    # -------------------------------------------------
    # 4. 데이터 증강
    # -------------------------------------------------
    "augmentation": {
        "use_synthetic": True,  # 합성 데이터 사용 여부
        "api_model": "",        # TODO: 합성 데이터 생성 모델 (예: "gpt-4o-mini")
        "augment_ratio": 0,     # TODO: 증강 비율 (예: 0.5 → 원본의 50% 추가 생성)
    },
    
    # -------------------------------------------------
    # 5. 분할
    # -------------------------------------------------
    "split": {
        "train_ratio": 0.9,     # 학습 데이터 비율
        "eval_ratio": 0.1,      # 평가 데이터 비율
    }
}

# 전략 요약 출력
def summarize_strategy(strategy):
    """데이터 전략을 요약합니다."""
    print("📊 데이터 전략 요약")
    print("="*50)
    print(f"  수집 소스: {strategy['collection']['sources']}")
    print(f"  목표 데이터 수: {strategy['collection']['target_count']}개")
    print(f"  데이터 형식: {strategy['format']['type']}")
    print(f"  최대 토큰: {strategy['format']['max_length']}")
    print(f"  합성 데이터: {'사용' if strategy['augmentation']['use_synthetic'] else '미사용'}")
    print(f"  Train/Eval: {strategy['split']['train_ratio']*100:.0f}% / {strategy['split']['eval_ratio']*100:.0f}%")

summarize_strategy(data_strategy)

### 📌 데이터 형식 예시

**Alpaca 형식** (Instruction Tuning에 적합)
```json
{
    "instruction": "다음 증상에 대해 가능한 질환을 알려주세요.",
    "input": "두통, 발열, 인후통이 3일째 지속됩니다.",
    "output": "말씀하신 증상(두통, 발열, 인후통)은 일반적으로 감기나 독감의 초기 증상일 수 있습니다..."
}
```

**ShareGPT 형식** (대화형에 적합)
```json
{
    "conversations": [
        {"from": "human", "value": "두통, 발열, 인후통이 3일째 지속됩니다."},
        {"from": "gpt", "value": "말씀하신 증상으로 보아 감기나 독감이 의심됩니다..."}
    ]
}
```

---
## 4️⃣ 모델 선택 가이드

RTX 4060 (8GB VRAM) 환경에서 선택 가능한 모델과 방법론입니다.

### 모델 크기별 가이드

| 모델 크기 | 방법론 | VRAM 사용량 | 학습 가능 여부 |
|----------|--------|------------|---------------|
| **1.5B** | FFT (Full Fine-Tuning) | ~7GB | ✅ 가능 |
| **1.5B** | LoRA | ~4GB | ✅ 매우 안정 |
| **3B** | LoRA | ~6GB | ✅ 가능 |
| **3B** | QLoRA (4bit) | ~4GB | ✅ 안정 |
| **7B** | QLoRA (4bit) | ~7GB | ⚠️ 빠듯 |
| **7B** | LoRA (FP16) | >8GB | ❌ 불가 |

### 추천 모델

| 모델 | 크기 | 한국어 성능 | 추천도 |
|------|------|-----------|--------|
| Qwen2.5-1.5B-Instruct | 1.5B | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ |
| Qwen2.5-3B-Instruct | 3B | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| Qwen2.5-7B-Instruct | 7B | ⭐⭐⭐⭐⭐ | ⭐⭐⭐ (QLoRA) |
| gemma-2-2b-it | 2B | ⭐⭐⭐ | ⭐⭐⭐ |
| Llama-3.2-1B-Instruct | 1B | ⭐⭐ | ⭐⭐⭐ |

In [ ]:
# =====================================================
# 📝 모델 선택 템플릿
# TODO: 아래 항목을 채워주세요
# =====================================================

model_config = {
    # -------------------------------------------------
    # 1. 베이스 모델
    # -------------------------------------------------
    "base_model": "",           # TODO: HuggingFace 모델 이름 (예: "Qwen/Qwen2.5-1.5B-Instruct")
    "model_size": "",           # TODO: 모델 크기 (예: "1.5B")
    
    # -------------------------------------------------
    # 2. 학습 방법론
    # -------------------------------------------------
    "method": "",               # TODO: "FFT", "LoRA", 또는 "QLoRA"
    
    # -------------------------------------------------
    # 3. LoRA 설정 (LoRA/QLoRA 사용 시)
    # -------------------------------------------------
    "lora_config": {
        "r": 16,                # LoRA rank (8, 16, 32, 64)
        "alpha": 32,            # LoRA alpha (보통 r*2)
        "dropout": 0.05,        # LoRA dropout
        "target_modules": [],   # TODO: 타겟 모듈 (예: ["q_proj", "v_proj"])
    },
    
    # -------------------------------------------------
    # 4. 학습 하이퍼파라미터
    # -------------------------------------------------
    "training": {
        "epochs": 3,            # 에포크 수
        "batch_size": 1,        # 배치 크기 (RTX 4060: 1~2)
        "grad_accum": 8,        # Gradient accumulation steps
        "learning_rate": 2e-4,  # 학습률
        "max_seq_length": 1024, # 최대 시퀀스 길이
    },
    
    # -------------------------------------------------
    # 5. 강화학습 (선택사항)
    # -------------------------------------------------
    "use_dpo": False,           # DPO 학습 사용 여부
}

# 설정 확인
def estimate_vram(config):
    """대략적인 VRAM 사용량을 추정합니다."""
    size_map = {"1B": 1, "1.5B": 1.5, "2B": 2, "3B": 3, "7B": 7}
    model_size = size_map.get(config["model_size"], 0)
    
    if config["method"] == "FFT":
        vram = model_size * 4  # FP16 weights + optimizer states
    elif config["method"] == "LoRA":
        vram = model_size * 2.5  # FP16 weights + LoRA
    elif config["method"] == "QLoRA":
        vram = model_size * 1.2  # 4bit weights + LoRA
    else:
        vram = 0
    
    print(f"📊 VRAM 추정: ~{vram:.1f}GB")
    if vram > 7.5:
        print("⚠️ RTX 4060 (8GB)에서 OOM 위험! 모델 크기를 줄이거나 QLoRA를 사용하세요.")
    elif vram > 6:
        print("⚠️ VRAM 여유가 적습니다. batch_size=1을 권장합니다.")
    else:
        print("✅ RTX 4060에서 안정적으로 학습 가능합니다.")
    return vram

if model_config["base_model"]:
    estimate_vram(model_config)
else:
    print("⚠️ base_model을 선택해주세요.")

---
## 5️⃣ 프로젝트 계획서 작성

지금까지 정한 내용을 종합하여 **프로젝트 계획서**를 작성합니다.

> 💡 **Hint**: 아래 딕셔너리의 빈 항목을 모두 채워주세요. 이 계획서는 이후 모든 노트북에서 참조됩니다.

In [ ]:
# =====================================================
# 📝 프로젝트 계획서 (최종)
# TODO: 모든 빈 항목을 채워주세요
# =====================================================

project_plan = {
    # === 기본 정보 ===
    "project_name": "",         # TODO: 프로젝트 이름 (예: "의료 상담 AI 어시스턴트")
    "team_members": [],         # TODO: 팀원 이름 (예: ["홍길동", "김철수"])
    "date": "2026-03-18",       # 작성일
    
    # === 문제 정의 (위에서 작성한 내용) ===
    "problem_definition": project_definition,
    
    # === 데이터 전략 (위에서 작성한 내용) ===
    "data_strategy": data_strategy,
    
    # === 모델 설정 (위에서 작성한 내용) ===
    "model_config": model_config,
    
    # === 기대 효과 ===
    "expected_outcome": "",     # TODO: 프로젝트 완료 시 기대되는 효과
    
    # === 리스크 및 대응 ===
    "risks": [
        # TODO: 예상되는 리스크와 대응 방안 추가
        # 예: {"risk": "데이터 부족", "mitigation": "합성 데이터로 보완"}
    ],
}

# 계획서 출력
def print_project_plan(plan):
    """프로젝트 계획서를 보기 좋게 출력합니다."""
    print("\n" + "="*60)
    print(f"📋 프로젝트 계획서: {plan['project_name'] or '(미작성)'}")
    print("="*60)
    print(f"\n👥 팀원: {', '.join(plan['team_members']) if plan['team_members'] else '(미작성)'}")
    print(f"📅 작성일: {plan['date']}")
    
    pd = plan['problem_definition']
    print(f"\n🎯 도메인: {pd['problem']['domain'] or '(미작성)'}")
    print(f"📌 태스크: {pd['problem']['target_task'] or '(미작성)'}")
    print(f"🎯 목표: {pd['goal']['primary'] or '(미작성)'}")
    
    mc = plan['model_config']
    print(f"\n🤖 모델: {mc['base_model'] or '(미작성)'}")
    print(f"🔧 방법: {mc['method'] or '(미작성)'}")
    
    ds = plan['data_strategy']
    print(f"\n📊 데이터 목표: {ds['collection']['target_count']}개")
    print(f"📊 형식: {ds['format']['type'] or '(미작성)'}")
    
    print(f"\n🌟 기대 효과: {plan['expected_outcome'] or '(미작성)'}")
    
    if plan['risks']:
        print(f"\n⚠️ 리스크:")
        for r in plan['risks']:
            print(f"   - {r['risk']} → {r['mitigation']}")
    
    print("\n" + "="*60)

print_project_plan(project_plan)

In [ ]:
# =====================================================
# 💾 프로젝트 계획서 저장
# 이후 노트북에서 이 파일을 불러와 사용합니다
# =====================================================

import json
import os

# 프로젝트 디렉토리 생성
project_dir = "./my_project"  # TODO: 원하는 프로젝트 디렉토리명으로 변경
os.makedirs(project_dir, exist_ok=True)
os.makedirs(f"{project_dir}/data", exist_ok=True)
os.makedirs(f"{project_dir}/models", exist_ok=True)
os.makedirs(f"{project_dir}/results", exist_ok=True)
os.makedirs(f"{project_dir}/app", exist_ok=True)

# 계획서 저장
plan_path = f"{project_dir}/project_plan.json"
with open(plan_path, "w", encoding="utf-8") as f:
    json.dump(project_plan, f, ensure_ascii=False, indent=2)

print(f"✅ 프로젝트 계획서가 저장되었습니다: {plan_path}")
print(f"\n📁 프로젝트 구조:")
for root, dirs, files in os.walk(project_dir):
    level = root.replace(project_dir, "").count(os.sep)
    indent = " " * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = " " * 2 * (level + 1)
    for file in files:
        print(f"{subindent}{file}")

---
## 6️⃣ 타임라인 및 체크리스트

### 프로젝트 타임라인

| 일차 | 단계 | 주요 활동 | 산출물 |
|------|------|----------|--------|
| **Day 1** | 기획 | 도메인 선택, 문제 정의, 데이터 전략 | 프로젝트 계획서 |
| **Day 2** | 데이터 | 데이터 수집/정제/증강 파이프라인 | 학습용 데이터셋 |
| **Day 3-4** | 학습 | LoRA/QLoRA 학습, 하이퍼파라미터 튜닝 | 파인튜닝된 모델 |
| **Day 4** | 평가 | 자동/수동 평가, 개선 전략 수립 | 평가 보고서 |
| **Day 5-6** | 배포 | 양자화, API 서버, Streamlit 앱 | 배포된 어플리케이션 |
| **Day 6** | 발표 | 최종 발표, 프로젝트 회고 | 발표 자료 |

In [ ]:
# =====================================================
# ✅ 프로젝트 체크리스트
# 각 단계별 완료 여부를 체크하세요
# =====================================================

checklist = {
    "1단계: 기획": {
        "도메인 선택 완료": False,          # TODO: 완료 시 True로 변경
        "문제 정의 완료": False,
        "데이터 전략 수립 완료": False,
        "모델 선택 완료": False,
        "프로젝트 계획서 저장 완료": False,
    },
    "2단계: 데이터": {
        "데이터 수집 완료": False,
        "데이터 정제 완료": False,
        "데이터 형식 변환 완료": False,
        "데이터 검증 완료": False,
        "Train/Eval 분할 완료": False,
    },
    "3단계: 학습": {
        "모델 로딩 확인": False,
        "1차 학습 완료": False,
        "학습 로그 분석 완료": False,
        "하이퍼파라미터 튜닝": False,
        "최종 모델 저장": False,
    },
    "4단계: 평가": {
        "자동 평가 완료": False,
        "LLM-as-a-Judge 평가 완료": False,
        "정성 평가 완료": False,
        "개선 전략 수립": False,
        "최종 성능 보고서 작성": False,
    },
    "5단계: 배포": {
        "모델 양자화 완료": False,
        "API 서버 구현": False,
        "Streamlit 앱 구현": False,
        "데모 시나리오 준비": False,
        "최종 발표 준비": False,
    },
}

# 체크리스트 출력
def print_checklist(cl):
    total = 0
    done = 0
    for phase, items in cl.items():
        phase_done = sum(1 for v in items.values() if v)
        phase_total = len(items)
        total += phase_total
        done += phase_done
        status = "✅" if phase_done == phase_total else "🔲"
        print(f"\n{status} {phase} ({phase_done}/{phase_total})")
        for item, completed in items.items():
            mark = "✅" if completed else "⬜"
            print(f"   {mark} {item}")
    
    print(f"\n{'='*40}")
    print(f"전체 진행률: {done}/{total} ({done/total*100:.0f}%)")
    progress = int(done/total * 20)
    print(f"[{'█' * progress}{'░' * (20-progress)}]")

print_checklist(checklist)

---
## 📝 정리

### 이번 세션에서 완료한 것
- ✅ 프로젝트 도메인 선택
- ✅ 문제 정의 (문제, 목표, 평가 기준)
- ✅ 데이터 전략 수립 (수집 방법, 양, 형식)
- ✅ 모델 및 학습 방법론 선택
- ✅ 프로젝트 계획서 작성 및 저장

### 다음 세션 준비사항
- 📌 `project_plan.json` 파일이 저장되어 있는지 확인
- 📌 데이터 소스 URL/경로를 미리 조사해 두기
- 📌 HuggingFace에서 관련 데이터셋 검색해 두기

### 다음 노트북
👉 **33_project_data_pipeline.ipynb**: 프로젝트 데이터 파이프라인 구축